# Recode Genotypes

## Objetive:

Generate a functional dataframe to run ML models

In [6]:
import pandas as pd
import numpy as np

In [107]:
gens = pd.read_excel("../data/genotypes.xlsx")

gens.dropna(subset="ID", inplace=True)

# Select interest variable, covariates, and predictors
genotypes_cols = [col for col in gens.columns[8:] if "MBL" not in col]

cols = ["ETNIA", "SEXO", "IDADE", "ELISA_IgG"] + genotypes_cols

model_data = gens.loc[:, cols]
model_data = model_data.reset_index(names="AMOSTRA")
model_data.head()

,AMOSTRA,ETNIA,SEXO,IDADE,ELISA_IgG,IL17_rs2275913G_A,IL17_rs4711998A_G,FAS670_rs1800682G_A,FOXP3_rs3761548G_T,IL4_rs2243250T_C,IL10_rs1800896T_C,ACE1,ACE2_rs2285666G_A,ACE2_rs73635825A_G,TMPRSS2_rs12329760C_T
0,0,1.0,Feminino,69.0,Não Reagente,NaN,NaN,AG,GG,TT,TT,ID,GG,AG,CC
1,1,1.0,Masculino,71.0,Não Reagente,GG,AG,AG,G,TT,CT,ID,G,A,CC
2,2,1.0,Masculino,69.0,Reagente,GG,AG,AA,NaN,TT,TT,II,GG,A,CC
3,3,1.0,Feminino,12.0,Não Reagente,GG,AG,AG,GG,TT,TT,ID,GA,AG,CC
4,4,1.0,Masculino,4.0,Não Reagente,GG,AG,AG,G,TT,TT,II,G,A,CC


In [108]:
# Recode sex
model_data["SEXO"] = model_data["SEXO"].map({"Masculino": 1, "Feminino": 0})

# Recode ELISA
model_data["ELISA_IgG"] = model_data["ELISA_IgG"].map({"Reagente": 1, "Não Reagente": 0})

# Standardize genotypes
model_data[genotypes_cols] = model_data[genotypes_cols].apply(lambda x: x.str.strip())

In [109]:
coding_gen = pd.read_excel("../data/genotypes_coding_models.xlsx")

long_data = (
    model_data
    .melt(
        id_vars=["AMOSTRA", "ETNIA", "SEXO", "IDADE", "ELISA_IgG"],
        var_name="VARIANTE",
        value_name="GENOTIPO"
    )
)

recoded_genotypes = long_data.merge(coding_gen, on=["VARIANTE", "GENOTIPO"])

recoded_genotypes.drop(columns="GENOTIPO", inplace=True)
recoded_genotypes.to_csv("../data/recoded_genotypes.csv", index=False)

recoded_genotypes.head()

,AMOSTRA,ETNIA,SEXO,IDADE,ELISA_IgG,VARIANTE,MODELO,CODIGO
0,1,1.0,1,71.0,0,IL17_rs2275913G_A,Additive,0
1,1,1.0,1,71.0,0,IL17_rs2275913G_A,Dominant,0
2,1,1.0,1,71.0,0,IL17_rs2275913G_A,Recessive,0
3,2,1.0,1,69.0,1,IL17_rs2275913G_A,Additive,0
4,2,1.0,1,69.0,1,IL17_rs2275913G_A,Dominant,0
